![WEkEO by Copernicus and Partners](./WEkEO_banner_EUMETSAT.png)

Copyright (c) 2025 European Union 

License: MIT

# Tutorial: Downloading CLMS Grassland Data with the WEkEO HDA Client


**This notebook stems from the wekeo4land repository (https://github.com/wekeo/wekeo4land) - if you face permission issues while running the notebook, clone it to your WEkEO workspace.*

In this tutorial, we use the WEkEO JupyterHub and the Harmonised Data Access (HDA) API to search for and download data from the [Copernicus Land Monitoring Service (CLMS)](https://land.copernicus.eu/).

The example query requests the **CLMS Grassland 10 m yearly product** for **2024** over the bounding box defined below. The workflow covers HDA authentication, dataset search, download, optional ZIP extraction, and workspace cleanup.


**PREREQUISITES**

This notebook has the following prerequisites:
- A WEkEO account.
- Access to the WEkEO JupyterHub or another Jupyter environment.
- WEkEO HDA credentials configured in a `.hdarc` file in your home directory.

**LEARNING OUTCOMES**

At the end of this notebook you will know how to:
- identify the required dataset information in WEkEO;
- define a CLMS HDA query;
- search for and download data with the WEkEO HDA client;
- decompress downloaded data;
- clean up downloaded and processed files.

**CONTENTS**

1. [Prepare your environment](#1-prepare-your-environment)
2. [Process data with HDA Client](#2-process-data-with-hda-client)
3. [Clean up](#3-clean-up)
4. [Data reference](#4-data-reference)

<hr>

## 1. Prepare your environment

### Environment Setup
Before we begin, we need to prepare our environment by installing and importing the necessary Python libraries:

In [3]:
#Uncomment and run if necessary
!pip install hda mypy

  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 58.2 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.7/531.7 kB 6.8 MB/s  0:00:00
Using cached mypy_extensions-1.1.0-py3-none-any.whl (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [mypy]4/5 [mypy]erialize]


In [4]:
# Load all required libraries

# Built-in modules
import os
import shutil
import warnings
from zipfile import ZipFile
from typing import Union

# Data handling and geospatial processing
import geopandas as gpd
import numpy as np
from rasterio.features import rasterize
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
import mypy

# Visualisation
import folium

# HDA API client
from hda import Client

# Notebook configuration
warnings.filterwarnings("ignore")

### WEkEO Account Registration

If you don't have a WEkEO account, please self-register at the <a href='https://wekeo.copernicus.eu/register' target='_blank'>WEkEO registration page</a>.

### HDA API Authentication

In order to interact with WEkEO's Harmonised Data Access API, each user shall ensure that the file '.hdarc' with username and password exists in the home directory. Please, find the tutorial on "how to" <a href='https://help.wekeo.eu/en/articles/6751608-how-to-use-the-hda-api-in-python' target='_blank'>here</a>. 

<hr>

## 2. Process data with HDA Client

### Search for the Dataset ID from the WEkEO Landing Platform

The WEkEO data catalogue provides the information needed to build an HDA request, including the dataset ID, temporal coverage, spatial coverage, and dataset-specific metadata.

For this notebook, the query is already configured for the **CLMS Grassland 10 m yearly product (2017–present)**. You can adapt the bounding box and dates in the next cell to request a different area or year supported by the dataset.


In [5]:
# Query for CLMS Grassland 10 m yearly data
query_CLMS = {
    "dataset_id": "EO:CLMS:DAT:GRA_10M_YEARLY_2017_PRESENT",
    "bbox": [
        6.350667697456856,
        54.31318755904486,
        13.265810732724411,
        58.25052763024132
    ],
    "startdate": "2024-01-01T00:00:00.000Z",
    "enddate": "2024-12-31T23:59:59.999Z",
    "datasetIdentifier": [
        "clms_vlcc_grassland_europe_10m_yearly_v1"
    ],
    "itemsPerPage": 200,
    "startIndex": 0
}

### Download Requested Data

You can use the client directly to download the data, as shown in the following example.

In [9]:
hda_client = Client()

# Store downloads in a local folder relative to the notebook
download_dir_path = os.path.join(os.getcwd(), 'data', 'download', 'land')
os.makedirs(download_dir_path, exist_ok=True)

# Search first — do not download yet
matches = hda_client.search(query_CLMS)

print(f"Found {len(matches)} result(s):\n")

result_ids = []

for i, item in enumerate(matches):

    # Each item is a SearchResults object containing one result
    result = item.results[0]

    # Product ID
    product_id = result["id"]
    result_ids.append(product_id)

    # Useful metadata
    properties = result.get("properties", {})
    metadata = properties.get("metadata", {})

    nominal_date = metadata.get("nominalDate", "N/A")
    cell_id = metadata.get("cellID", "N/A")
    size_mb = properties.get("size", 0) / (1024 ** 2)

    print(f"[{i:2d}] {product_id}")
    print(f"     Date: {nominal_date}")
    print(f"     Cell: {cell_id}")
    print(f"     Size: {size_mb:.2f} MB")
    print()

Found 28 result(s):

[ 0] CLMS_HRLVLCC_GRA_S2024_R10m_E43N38_03035_V01_R00
     Date: 2024-01-01T00:00:00.000000Z
     Cell: 100kmE43N38
     Size: 1.78 MB

[ 1] CLMS_HRLVLCC_GRA_S2024_R10m_E42N35_03035_V01_R00
     Date: 2024-01-01T00:00:00.000000Z
     Cell: 100kmE42N35
     Size: 4.64 MB

[ 2] CLMS_HRLVLCC_GRA_S2024_R10m_E42N37_03035_V01_R00
     Date: 2024-01-01T00:00:00.000000Z
     Cell: 100kmE42N37
     Size: 4.21 MB

[ 3] CLMS_HRLVLCC_GRA_S2024_R10m_E41N39_03035_V01_R00
     Date: 2024-01-01T00:00:00.000000Z
     Cell: 100kmE41N39
     Size: 5.38 MB

[ 4] CLMS_HRLVLCC_GRA_S2024_R10m_E41N34_03035_V01_R00
     Date: 2024-01-01T00:00:00.000000Z
     Cell: 100kmE41N34
     Size: 0.76 MB

[ 5] CLMS_HRLVLCC_GRA_S2024_R10m_E41N38_03035_V01_R00
     Date: 2024-01-01T00:00:00.000000Z
     Cell: 100kmE41N38
     Size: 1.05 MB

[ 6] CLMS_HRLVLCC_GRA_S2024_R10m_E42N38_03035_V01_R00
     Date: 2024-01-01T00:00:00.000000Z
     Cell: 100kmE42N38
     Size: 0.67 MB

[ 7] CLMS_HRLVLCC_GRA_S2024

In [17]:
# Choose which results to download by their list indices.
# Examples:
selected_indices = [0]                       # first result only
# selected_indices = [0, 2, 5]              # several results
#selected_indices = list(range(len(matches)))  # all results


# Validate the selection
invalid_indices = [
    i for i in selected_indices
    if i < 0 or i >= len(matches)
]

if invalid_indices:
    raise IndexError(
        f"Invalid result index/indices: {invalid_indices}. "
        f"Choose values between 0 and {len(matches) - 1}."
    )


# Display selected products
print(f"Selected {len(selected_indices)} product(s) for download:\n")

for i in selected_indices:

    result = matches[i].results[0]

    product_id = result["id"]

    properties = result.get("properties", {})
    metadata = properties.get("metadata", {})

    cell_id = metadata.get("cellID", "N/A")
    nominal_date = metadata.get("nominalDate", "N/A")
    size_mb = properties.get("size", 0) / (1024 ** 2)

    print(f"[{i:2d}] {product_id}")
    print(f"     Cell: {cell_id}")
    print(f"     Date: {nominal_date}")
    print(f"     Size: {size_mb:.2f} MB")
    print()

Selected 1 product(s) for download:

[ 0] CLMS_HRLVLCC_GRA_S2024_R10m_E43N38_03035_V01_R00
     Cell: 100kmE43N38
     Date: 2024-01-01T00:00:00.000000Z
     Size: 1.78 MB



In [18]:
# Download only the selected results.
for i in selected_indices:
    print(f"\nDownloading result [{i}] ...")
    matches[i].download(download_dir_path)

print("\nDownload complete.")


Download complete.


### Decompress downloaded ZIP files


In [19]:
processing_dir_path = os.path.join(os.getcwd(), 'data', 'processing', 'land')
os.makedirs(processing_dir_path, exist_ok=True)

zip_files = [name for name in os.listdir(download_dir_path) if name.lower().endswith('.zip')]

if not zip_files:
    print('No ZIP files found. The downloaded products may already be in their final format.')
else:
    for item in zip_files:
        file_name = os.path.join(download_dir_path, item)
        print(f'Decompressing {item} ... ', end='')
        with ZipFile(file_name, 'r') as zip_ref:
            zip_ref.extractall(processing_dir_path)
        print('DONE')


Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E45N39_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E41N34_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E45N35_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E44N34_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E42N36_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E41N36_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E42N34_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E43N36_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E42N35_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E43N37_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E45N37_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E45N34_03035_V01_R00.zip ... DONE
Decompressing CLMS_HRLVLCC_GRA_S2024_R10m_E42N39_03035_V01_R00.zip ... DONE
Decompressin

<hr>

## 3. Clean up

To ensure a clean workspace and remove all downloaded files and processing artifacts created during this session, run the following code. This will delete any files that were downloaded and processed within this notebook.

In [ ]:
paths_to_cleanup = [
    download_dir_path,
    processing_dir_path
]

for path in paths_to_cleanup:
    if os.path.isfile(path):
        os.remove(path)
    elif os.path.isdir(path):
        shutil.rmtree(path)

print("Cleanup complete. All downloaded and processed files have been removed.")

<hr>

## 4. Data reference

This notebook uses the **CLMS Grassland 10 m yearly product (2017–present)** through WEkEO, with dataset ID `EO:CLMS:DAT:GRA_10M_YEARLY_2017_PRESENT` and dataset identifier `clms_vlcc_grassland_europe_10m_yearly_v1`, as specified by the HDA query used above.

For authoritative product metadata and citation information, consult the corresponding dataset entry in the WEkEO catalogue before publication or redistribution.


<p><img src='../../media/land/all_partners_wekeo_2.png' align='left' alt='Logo EU Copernicus' height='400px'></img></p>

[View on GitHub](https://github.com/wekeo) | 
[Visit WEkEO](https://wekeo.copernicus.eu) | [Run Notebooks on the WEkEO Workspace](https://jupyterhub.prod.wekeo2.eu/) | 
[Contact helpdesk for support](mailto:support@wekeo.eu)

